# 卷积神经网络练习参考答案（第 6 章）

本文件是配套练习题的参考答案，请先在手写练习后再对照。

# 卷积神经网络练习（第 6 章）



- CNN 概述（卷积层、池化层、全连接层的作用）
- 卷积层：卷积运算、填充、步幅、3 维数据的卷积运算、`nn.Conv2d` 的使用
- 池化层：Max 池化与 Average 池化、`nn.MaxPool2d` / `nn.AvgPool2d`
- 应用案例：Fashion-MNIST 服装分类（加载数据、搭建模型、模型训练）

说明：本练习在教材示例的基础上做了适当调整（卷积核大小、通道数、激活函数、初始化方式、
优化器等均与教材不同），请先阅读题目描述，再在下方代码单元格中**手写代码**完成练习，
写完后与 `answer` 目录下的答案对照。

部分练习所需的数据位于项目根目录的 `data/` 下，本 notebook 中使用相对路径 `../../data/...`。

先执行下面的单元格导入所需的库。

In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
np.set_printoptions(precision=3, suppress=True)

print("torch version:", torch.__version__)

## 6.1 卷积层

卷积层对数据进行卷积运算：以一定间隔滑动卷积核的窗口，将各个位置上卷积核的元素和输入的
对应元素相乘再求和（乘积累加）。卷积核的参数相当于权重，此外还有偏置。

卷积层接收 3 维形状（通道, 高, 宽）的输入，同样以 3 维形状输出，因此不会像全连接层那样
丢失空间信息。

### 练习 1：手写单通道二维卷积运算

请实现一个不借助 `F.conv2d` 的卷积函数，并用它计算下面 5×5 输入与 3×3 卷积核的卷积结果。

要求：
1. 实现 `conv2d_naive(x, w, b=None, stride=1, padding=0)`，其中 `x` 形状为 `(C, H, W)`，
   `w` 形状为 `(FN, C, FH, FW)`，返回形状 `(FN, OH, OW)`；`padding` 用 0 填充；
2. 输入使用下面的 `x`（先 `unsqueeze` 成 `(1, 5, 5)`）、卷积核使用 `w`
   （先 `unsqueeze` 成 `(1, 1, 3, 3)`），步幅 1、无填充，打印卷积结果；
3. 用 `F.conv2d` 计算同样的卷积，验证你的结果与之相等（`torch.allclose`）。

In [ ]:
# 练习 1：手写单通道二维卷积运算

def conv2d_naive(x, w, b=None, stride=1, padding=0):
    C, H, W = x.shape
    FN, _, FH, FW = w.shape
    if padding:
        x = F.pad(x, (padding, padding, padding, padding))
        C, H, W = x.shape
    OH = (H - FH) // stride + 1
    OW = (W - FW) // stride + 1
    out = torch.zeros(FN, OH, OW)
    for fn in range(FN):
        for i in range(OH):
            for j in range(OW):
                region = x[:, i * stride:i * stride + FH, j * stride:j * stride + FW]
                out[fn, i, j] = (region * w[fn]).sum()
        if b is not None:
            out[fn] += b[fn]
    return out


x = torch.tensor([[1., 2., 3., 0., 1.],
                  [0., 1., 2., 3., 1.],
                  [1., 0., 1., 2., 0.],
                  [2., 1., 0., 1., 3.],
                  [1., 3., 2., 1., 0.]])
w = torch.tensor([[1., 0., -1.],
                  [1., 0., -1.],
                  [1., 0., -1.]])

out = conv2d_naive(x.unsqueeze(0), w.unsqueeze(0).unsqueeze(0))
print("手写卷积结果:\n", out)

ref = F.conv2d(x.view(1, 1, 5, 5), w.view(1, 1, 3, 3))[0]
print("torch 官方结果:\n", ref)
print("两者是否一致:", torch.allclose(out, ref))

### 练习 2：填充（padding）

填充是在输入数据周围填入固定数据（通常为 0），用于调整输出数据的形状大小。

请基于练习 1 的 `conv2d_naive`，对形状为 `(1, 4, 4)`、元素全为 1 的输入数据，
使用 3×3、元素全为 1 的卷积核，分别计算：

1. 不填充（`padding=0`）时的输出形状；
2. 填充幅度为 1（`padding=1`）时的输出形状。

打印两种情况的输出形状，并说明填充是如何影响输出形状的。

In [ ]:
# 练习 2：填充对输出形状的影响

conv2d_naive = conv2d_naive  # 使用练习 1 中实现的函数

x = torch.ones(1, 4, 4)
w = torch.ones(1, 1, 3, 3)

out0 = conv2d_naive(x, w, padding=0)
out1 = conv2d_naive(x, w, padding=1)
print("padding=0 输出形状:", tuple(out0.shape))
print("padding=1 输出形状:", tuple(out1.shape))

# 验证
print("padding=0 与官方一致:", torch.allclose(out0, F.conv2d(x.unsqueeze(0), w)[0]))
print("padding=1 与官方一致:", torch.allclose(out1, F.conv2d(x.unsqueeze(0), w, padding=1)[0]))

### 练习 3：步幅（stride）

步幅是应用卷积核的位置间隔。请基于练习 1 的 `conv2d_naive`，对形状为 `(1, 4, 4)`
的输入（元素全为 1）应用幅度为 1 的填充，并使用 3×3 的卷积核、**步幅为 3** 进行卷积：

1. 打印输出的形状（想一想为什么是 2×2）；
2. 用 `F.conv2d` 验证结果一致。

In [ ]:
# 练习 3：步幅的卷积运算

x = torch.ones(1, 4, 4)
w = torch.ones(1, 1, 3, 3)

out = conv2d_naive(x, w, stride=3, padding=1)
print("步幅为 3 的输出形状:", tuple(out.shape))

ref = F.conv2d(x.unsqueeze(0), w, stride=3, padding=1)[0]
print("与官方一致:", torch.allclose(out, ref))
# OH = (4 + 2*1 - 3) // 3 + 1 = 2

### 练习 4：输出尺寸计算公式

假设输入为 `(H, W)`，卷积核为 `(FH, FW)`，填充为 `P`，步幅为 `S`，则输出尺寸为：

$$OH=\frac{H+2P-FH}{S}+1$$

请实现函数 `conv_output_size(H, FH, P, S)`（结果为向下取整，与 PyTorch 保持一致），
并计算下面几组参数对应的输出边长：

| H | FH | P | S |
|---|----|---|---|
| 4 | 3 | 1 | 3 |
| 28 | 5 | 2 | 1 |
| 28 | 3 | 1 | 2 |
| 7 | 3 | 0 | 1 |

In [ ]:
# 练习 4：实现输出尺寸计算公式

def conv_output_size(H, FH, P, S):
    return (H + 2 * P - FH) // S + 1


cases = [(4, 3, 1, 3), (28, 5, 2, 1), (28, 3, 1, 2), (7, 3, 0, 1)]
for H, FH, P, S in cases:
    print(f"H={H}, FH={FH}, P={P}, S={S} -> OH={conv_output_size(H, FH, P, S)}")

# 用 nn.Conv2d 实际验证其中一组
conv = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1)
y = conv(torch.randn(1, 1, 28, 28))
print("Conv2d 实测输出:", tuple(y.shape), "公式计算:", conv_output_size(28, 3, 1, 2))

### 练习 5：3 维数据（多通道）的卷积运算

图像是 3 维数据（通道, 高, 宽）。在 3 维卷积中，输入数据的通道数与卷积核的通道数必须
相同；使用 `FN` 个卷积核就能得到 `FN` 个输出通道。

请基于练习 1 的 `conv2d_naive`：

1. 用 `torch.randn` 构造形状为 `(3, 5, 5)` 的输入 `x` 和形状为 `(2, 3, 3, 3)` 的卷积核 `w`、
   形状为 `(2,)` 的偏置 `b`（`torch.manual_seed(42)` 保证可复现）；
2. 计算卷积，打印输出形状（应为 `(2, 3, 3)`）；
3. 用 `F.conv2d`（输入需 `unsqueeze` 成 `(1, 3, 5, 5)`）验证结果一致。

In [ ]:
# 练习 5：3 维数据（多通道）的卷积运算

torch.manual_seed(42)
x = torch.randn(3, 5, 5)
w = torch.randn(2, 3, 3, 3)
b = torch.randn(2)

out = conv2d_naive(x, w, b, stride=1, padding=0)
print("输出形状:", tuple(out.shape))

ref = F.conv2d(x.unsqueeze(0), w, b)[0]
print("与官方一致:", torch.allclose(out, ref, atol=1e-6))

### 练习 6：`nn.Conv2d` 的参数与输出形状

`nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`：
`in_channels` 为输入通道数，`out_channels` 为输出通道数（卷积核个数），
`kernel_size` 为卷积核大小，`stride` 为步幅，`padding` 为填充幅度。

请构造卷积层 `nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2)`，
对形状为 `(1, 3, 32, 32)` 的输入做卷积：

1. 用练习 4 的公式计算输出的高和宽，再打印实际输出形状验证；
2. 计算该卷积层的参数总数（权重元素个数 + 偏置个数），并与
   `sum(p.numel() for p in conv.parameters())` 的结果比较。

In [ ]:
# 练习 6：Conv2d 参数与输出形状计算

conv = nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2)
x = torch.randn(1, 3, 32, 32)
y = conv(x)
print("实际输出形状:", tuple(y.shape))

OH = conv_output_size(32, 5, 2, 2)
OW = conv_output_size(32, 5, 2, 2)
print("公式计算的高、宽:", OH, OW)

# 权重形状 (out_channels, in_channels, FH, FW)，偏置形状 (out_channels,)
print("权重形状:", tuple(conv.weight.shape), "偏置形状:", tuple(conv.bias.shape))
manual = 16 * 3 * 5 * 5 + 16
actual = sum(p.numel() for p in conv.parameters())
print("手工计算参数量:", manual, " 程序统计参数量:", actual, " 是否一致:", manual == actual)

### 练习 7：用 `nn.Conv2d` 处理真实图片

请读取 `../../data/duck.jpg`，将其转换为张量并调整为 `(C, H, W)` 的形状，
然后使用 `nn.Conv2d(in_channels=3, out_channels=3, kernel_size=5, stride=4, padding=1, bias=False)`
进行卷积（注意教材用的是 9×9 卷积核、步幅 3，这里请按上面的参数实现）：

1. 打印输入、输出特征图的形状；
2. 将输出特征图转换回图片并可视化（原图与输出图并排显示）。

In [ ]:
# 练习 7：用 Conv2d 处理真实图片

img = plt.imread("../../data/duck.jpg")
print("图片数据形状:", img.shape)

x = torch.tensor(img).permute(2, 0, 1).float()
print("输入特征图形状:", x.shape)

conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=5, stride=4, padding=1, bias=False)
y = conv(x)
print("输出特征图形状:", y.shape)

out_img = torch.clamp(y.int(), 0, 255).permute(1, 2, 0).detach().numpy()
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img)
ax[0].set_title("原图")
ax[1].imshow(out_img)
ax[1].set_title("卷积输出")
for a in ax:
    a.axis("off")
plt.show()

## 6.2 池化层

池化层通过缩小长、宽方向上的空间来降维，能够缩减模型大小、提高计算速度。常见的池化有
Max 池化（取窗口内最大值）和 Average 池化（取窗口内平均值）。

池化层**没有要学习的参数**，且池化运算**按通道独立进行**，经过池化后通道数不变。

### 练习 8：手写 Max 池化

请实现 `maxpool_naive(x, k=2, s=2)`，`x` 形状为 `(C, H, W)`，返回形状 `(C, OH, OW)`，
对每个通道独立地在 `k×k` 窗口内取最大值：

1. 用 `torch.manual_seed(0)` 构造形状为 `(1, 4, 4)` 的随机输入；
2. 用你的函数做 2×2、步幅 2 的 Max 池化，打印输出形状；
3. 用 `nn.MaxPool2d(kernel_size=2, stride=2)` 验证结果一致。

In [ ]:
# 练习 8：手写 Max 池化并与 nn.MaxPool2d 对比

def maxpool_naive(x, k=2, s=2):
    C, H, W = x.shape
    OH = (H - k) // s + 1
    OW = (W - k) // s + 1
    out = torch.zeros(C, OH, OW)
    for c in range(C):
        for i in range(OH):
            for j in range(OW):
                out[c, i, j] = x[c, i * s:i * s + k, j * s:j * s + k].max()
    return out


torch.manual_seed(0)
x = torch.randn(1, 4, 4)

out = maxpool_naive(x, k=2, s=2)
print("输出形状:", tuple(out.shape))
print("手写 Max 池化结果:\n", out)

ref = nn.MaxPool2d(kernel_size=2, stride=2)(x)
print("与 nn.MaxPool2d 一致:", torch.allclose(out, ref))

### 练习 9：手写 Average 池化

请实现 `avgpool_naive(x, k=2, s=2)`，对每个通道独立地在 `k×k` 窗口内取平均值：

1. 使用与练习 8 相同的随机输入（`(1, 4, 4)`，2×2、步幅 2）；
2. 打印 Average 池化的结果；
3. 用 `nn.AvgPool2d(kernel_size=2, stride=2)` 验证结果一致。

In [ ]:
# 练习 9：手写 Average 池化并与 nn.AvgPool2d 对比

def avgpool_naive(x, k=2, s=2):
    C, H, W = x.shape
    OH = (H - k) // s + 1
    OW = (W - k) // s + 1
    out = torch.zeros(C, OH, OW)
    for c in range(C):
        for i in range(OH):
            for j in range(OW):
                out[c, i, j] = x[c, i * s:i * s + k, j * s:j * s + k].mean()
    return out


torch.manual_seed(0)
x = torch.randn(1, 4, 4)

out = avgpool_naive(x, k=2, s=2)
print("Average 池化结果:\n", out)

ref = nn.AvgPool2d(kernel_size=2, stride=2)(x)
print("与 nn.AvgPool2d 一致:", torch.allclose(out, ref))

### 练习 10：池化的鲁棒性

池化的一个特点是对微小偏差具有鲁棒性：当数据发生微小偏差（如向宽度方向平移 1 个元素）时，
池化的输出可能保持不变。

请构造输入 `x`（形状 `(1, 4, 4)`，每一行都是 `[1, 9, 9, 9]`），并对它以及
"整体向左平移 1 个元素、末尾补 0 后"的数据 `x_shift` 分别做 2×2、步幅 2 的 Max 池化，
比较两次的输出是否相同，并打印结果。

提示：平移可用 `torch.cat([x[:, :, 1:], torch.zeros_like(x[:, :, :1])], dim=-1)`。

In [ ]:
# 练习 10：验证池化对微小偏差的鲁棒性

x = torch.tensor([[1., 9., 9., 9.]]).repeat(1, 4, 1)  # (C=1, H=4, W=4)

x_shift = torch.cat([x[:, :, 1:], torch.zeros_like(x[:, :, :1])], dim=-1)

out = maxpool_naive(x, k=2, s=2)
out_shift = maxpool_naive(x_shift, k=2, s=2)

print("原始数据池化输出:\n", out)
print("平移后池化输出:\n", out_shift)
print("两次输出是否相同:", torch.equal(out, out_shift))

## 6.3 应用案例：服装分类

Fashion-MNIST 数据集中每个样本都是 28×28 的灰度图像，对应 10 个类别
（0 T恤/上衣，1 裤子，2 套头衫，3 连衣裙，4 外套，5 凉鞋，6 衬衫，7 运动鞋，8 包，9 靴子）。

注意：教材使用的 `fashion-mnist_train.csv` / `fashion-mnist_test.csv` 在本地 `data/` 下不可用，
本练习改用 `../../data/train.csv`（格式相同：第 1 列为标签，第 2~785 列为 784 个像素），
并自行按 8:2 划分训练集与测试集。

### 练习 11：加载数据并划分训练集/测试集

请读取 `../../data/train.csv`，完成以下步骤：

1. 将像素列转换为浮点张量并 reshape 成 `(N, 1, 28, 28)`，将标签列转换为 `torch.int64`；
2. 使用 `torch.randperm` 按 8:2 划分训练集与测试集（注意训练/测试的划分要基于同一个随机排列）；
3. 用 `TensorDataset` 封装为 `train_dataset`、`test_dataset`，打印各自的样本数；
4. 可视化训练集中的某一个样本（`cmap="gray"`），并打印它的标签。

提示：像素值范围是 0~255，可以除以 255 归一化，以便训练更稳定。

In [ ]:
# 练习 11：加载数据并划分训练集/测试集

data = pd.read_csv("../../data/train.csv")
print("原始数据形状:", data.shape)

X = torch.tensor(data.iloc[:, 1:].values, dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
y = torch.tensor(data.iloc[:, 0].values, dtype=torch.int64)
print("X 形状:", tuple(X.shape), " y 形状:", tuple(y.shape))

perm = torch.randperm(len(X))
split = int(len(X) * 0.8)
train_idx, test_idx = perm[:split], perm[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("训练集:", X_train.shape[0], " 测试集:", X_test.shape[0])

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

idx = 100
plt.imshow(X_train[idx, 0], cmap="gray")
plt.title(f"label = {y_train[idx].item()}")
plt.show()

### 练习 12：搭建卷积神经网络模型

请用 `nn.Sequential` 搭建如下结构的模型（与教材不同，这里使用 ReLU、Max 池化和 Dropout）：

| 层 | 说明 |
|----|------|
| Conv2d(1, 8, kernel_size=3, padding=1) | 输出 8 通道，尺寸不变 |
| ReLU | 激活 |
| MaxPool2d(2, 2) | 28 → 14 |
| Conv2d(8, 32, kernel_size=3, padding=1) | 输出 32 通道，尺寸不变 |
| ReLU | 激活 |
| MaxPool2d(2, 2) | 14 → 7 |
| Flatten | 拉平为 `32 × 7 × 7` |
| Linear(32 * 7 * 7, 128) | 全连接 |
| ReLU | 激活 |
| Dropout(0.3) | 随机失活 |
| Linear(128, 10) | 输出 10 个类别 |

搭建完成后，输入一个形状为 `(1, 1, 28, 28)` 的随机张量，逐层打印输出的形状，
确认与预期一致。

In [ ]:
# 练习 12：搭建卷积神经网络模型

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(8, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10),
)

X = torch.randn(1, 1, 28, 28)
for layer in model:
    X = layer(X)
    print(f"{layer.__class__.__name__:<12} output shape: {tuple(X.shape)}")

print("模型参数量:", sum(p.numel() for p in model.parameters()))

### 练习 13：编写模型训练函数

请实现 `train(model, train_dataset, test_dataset, lr, epoch_num, batch_size, device)`：

- 对卷积层和全连接层的权重使用 **He（Kaiming）正态初始化**
  （`nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")`）；
- 使用交叉熵损失 `nn.CrossEntropyLoss` 和 **Adam** 优化器（教材用的是 SGD）；
- 每个 epoch：在训练集上训练，记录**平均训练损失**和**训练准确率**；
  再在测试集上评估，记录**测试准确率**（评估时用 `torch.no_grad()`）；
- 返回三个列表：`train_loss_list`、`train_acc_list`、`test_acc_list`。

提示：`kaiming_normal_` 会直接修改权重张量，但对 `nn.Sequential` 中的层用 `model.apply`
遍历时，`nn.Flatten`、`nn.ReLU` 等没有 `weight`，需要先判断类型。

In [ ]:
# 练习 13：模型训练函数

def train(model, train_dataset, test_dataset, lr, epoch_num, batch_size, device):
    def init_weights(layer):
        if isinstance(layer, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

    model.apply(init_weights)
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_loss_list, train_acc_list, test_acc_list = [], [], []

    for epoch in range(epoch_num):
        # ---- 训练 ----
        model.train()
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        loss_accumulate, correct_accumulate = 0.0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            output = model(X)
            loss_value = loss_fn(output, y)
            optimizer.zero_grad()
            loss_value.backward()
            optimizer.step()

            loss_accumulate += loss_value.item()
            _, pred = output.max(1)
            correct_accumulate += pred.eq(y).sum().item()

        train_loss_list.append(loss_accumulate / len(train_loader))
        train_acc_list.append(correct_accumulate / len(train_dataset))

        # ---- 评估 ----
        model.eval()
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        correct_accumulate = 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                output = model(X)
                _, pred = output.max(1)
                correct_accumulate += pred.eq(y).sum().item()
        test_acc_list.append(correct_accumulate / len(test_dataset))

        print(f"epoch {epoch + 1:>2}/{epoch_num} loss:{train_loss_list[-1]:.4f} "
              f"train_acc:{train_acc_list[-1]:.4f} test_acc:{test_acc_list[-1]:.4f}")

    return train_loss_list, train_acc_list, test_acc_list

### 练习 14：训练模型并绘制曲线

请调用练习 13 的 `train` 函数训练练习 12 的模型：

- `device` 使用 `torch.device("cuda" if torch.cuda.is_available() else "cpu")`；
- 学习率 `lr=0.001`，`epoch_num=3`，`batch_size=128`；
- 将训练损失、训练准确率、测试准确率画在同一张图上
  （损失用左轴，准确率用右轴，或画成两张子图均可），并加上图例。

In [ ]:
# 练习 14：训练模型并绘制曲线

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

train_loss_list, train_acc_list, test_acc_list = train(
    model, train_dataset, test_dataset, lr=0.001, epoch_num=3, batch_size=128, device=device)

epochs = range(1, len(train_loss_list) + 1)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(epochs, train_loss_list, "r-o", label="train_loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss", color="r")
ax2 = ax1.twinx()
ax2.plot(epochs, train_acc_list, "b-s", label="train_acc")
ax2.plot(epochs, test_acc_list, "k--^", label="test_acc")
ax2.set_ylabel("accuracy")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
plt.title("Fashion-MNIST 训练过程")
plt.show()

## 6.4 深度卷积神经网络（思考题）

将网络层数加深可以更有效地提取层次信息。著名的深度卷积神经网络包括：

- **AlexNet**（2012）：5 个卷积层 + 3 个全连接层，使用 ReLU 与 Dropout；
- **VGG**（2014）：由多个卷积-池化层堆叠而成，常见 VGG-16 / VGG-19；
- **GoogleNet**（2014）：引入 Inception 结构，横向上使用多个不同大小的滤波器再合并；
- **ResNet**（2015）：引入"快捷结构"（残差连接），学习目标由 $h(x)$ 变为 $h(x)-x$，
  有效缓解深度网络的梯度消失问题。

### 练习 15：残差连接（思考 + 编码验证）

ResNet 的核心是残差连接：某一层的输出为 $y = h(x) + x$。

请编写代码验证：对于一个使用残差连接的简单模块，即使内部变换 $h$ 的权重很小，
输出相对于输入的梯度也不会消失。

要求：
1. 定义一个包含 `nn.Linear(8, 8)`（权重初始化为很小的值，如乘以 0.01）的残差模块，
   前向为 `x + self.fc(x)`；
2. 分别用"带残差"和"不带残差"（直接输出 `self.fc(x)`）两种方式，
   对输入 `x` 求 `y.sum()` 关于 `x` 的梯度；
3. 比较两种情况下梯度的范数，观察残差连接对梯度的影响。

In [ ]:
# 练习 15：思考题：残差连接与梯度传播

class Residual(nn.Module):
    def __init__(self, with_residual=True):
        super().__init__()
        self.fc = nn.Linear(8, 8)
        nn.init.normal_(self.fc.weight, std=0.01)
        nn.init.zeros_(self.fc.bias)
        self.with_residual = with_residual

    def forward(self, x):
        h = self.fc(x)
        return x + h if self.with_residual else h


x = torch.randn(4, 8, requires_grad=True)

for flag in [True, False]:
    layer = Residual(with_residual=flag)
    layer.fc.weight.data.copy_(torch.randn(8, 8) * 0.01)
    layer.fc.bias.data.zero_()
    y = layer(x)
    y.sum().backward()
    print(f"with_residual={flag}: 梯度范数 = {x.grad.norm().item():.4f}")
    x.grad = None